In [ ]:
import glob
import json

In [39]:
instances_full_path = [i for i in glob.glob(
    '/home/beno/projects/datasets-internal/sidon/data/Advantage2_system1.8/*/*/*/*/*') if 'json' in i]

instances_full_path.sort()
partial_names = [ '_'.join(name[len('/home/beno/projects/datasets-internal/sidon/data/'):-len('.json')].split('/')) for name in instances_full_path]

partial_names

['Advantage2_system1.8_send_8_m1_case0_beta0.0_J_log',
 'Advantage2_system1.8_send_8_m1_case0_beta0.0_J_qac',
 'Advantage2_system1.8_send_8_m1_case1_beta0.0_J_log',
 'Advantage2_system1.8_send_8_m1_case1_beta0.0_J_qac',
 'Advantage2_system1.8_send_8_m1_case2_beta0.0_J_log',
 'Advantage2_system1.8_send_8_m1_case2_beta0.0_J_qac',
 'Advantage2_system1.8_send_8_m1_case3_beta0.0_J_log',
 'Advantage2_system1.8_send_8_m1_case3_beta0.0_J_qac',
 'Advantage2_system1.8_send_8_m1_case4_beta0.0_J_log',
 'Advantage2_system1.8_send_8_m1_case4_beta0.0_J_qac',
 'Advantage2_system1.8_send_8_m1_case5_beta0.0_J_log',
 'Advantage2_system1.8_send_8_m1_case5_beta0.0_J_qac',
 'Advantage2_system1.8_send_8_m1_case6_beta0.0_J_log',
 'Advantage2_system1.8_send_8_m1_case6_beta0.0_J_qac',
 'Advantage2_system1.8_send_8_m1_case7_beta0.0_J_log',
 'Advantage2_system1.8_send_8_m1_case7_beta0.0_J_qac',
 'Advantage2_system1.8_send_8_m1_case8_beta0.0_J_log',
 'Advantage2_system1.8_send_8_m1_case8_beta0.0_J_qac',
 'Advantag

In [ ]:
from sidion_qubo import Ising

In [62]:
from functools import wraps
from typing import Tuple
import scipy as sp
import numpy as np
from numpy import typing as npt


def to_sparse(qubo: npt.ArrayLike):
    """Converts a 2D array to the expected sparse coo_matrix representation.

    Args:
        qubo: 2D list or numpy array

    Returns:
        dict: a sparse QUBO representation needed as input for the Solver. It is as a json serializable dict.
    """
    qubo_sparse_coo = sp.sparse.coo_matrix(qubo)
    return {
        'shape': qubo_sparse_coo.shape,
        'nnz': qubo_sparse_coo.nnz,
        'row': qubo_sparse_coo.row.tolist(),
        'col': qubo_sparse_coo.col.tolist(),
        'data': qubo_sparse_coo.data.tolist()
    }

def qubo_to_biqbin_representation(qubo) -> dict:
    """Converts a dense qubo represantation 2D array to the expected biqbin format of a json serializable 
    dict with 'qubo' key and a sparse qubo represantation as value.

    Args:
        qubo: 2D list or numpy array

    Returns:
        dict: json serializable dictionary that Biqbin can parse. Save to file and pass the path to DataGetterJson.
    """
    if not np.all(np.asarray(qubo) % 1 == 0):
        raise ValueError("All QUBO values need to be integers!")

    return {
        'qubo': to_sparse(qubo)
    }

In [69]:
import networkx as nx
import numpy as np
isings = ['J_log', 'J_qac']

def ising2qubo(ising):
    Q, nodes = nx.attr_matrix(ising, edge_attr='J')
    Q = np.triu(Q, 1)
    node_h = nx.get_node_attributes(ising, 'h')
    L = np.array([node_h[node] for node in nodes])
    
    QUBO = 4*Q
    
    offset = -L.sum() + Q.sum()
    np.fill_diagonal(QUBO, 2*L - 2*Q.sum(axis=0) - 2*Q.sum(axis=1))

    return QUBO, offset

for i, instance in enumerate(instances_full_path):
    with open(instance, 'r') as f:
        data = json.load(f)

    ising = Ising().deserialize(data)
    filename = '_'.join(instance[len('/home/beno/projects/datasets-internal/sidon/data/'):-len('.json')].split('/')) + '.json'

    qubo, offset = ising2qubo(ising)
    qubo *= 28
    qubo_rounded = qubo.round()
    if not np.allclose(qubo, qubo_rounded):
        print(f"Oh nooo {instance}")
        break
    
    # biqbin_data = qubo_to_biqbin_representation(qubo_rounded)
    # biqbin_data['offset'] = offset * 28
    # with open(f'/home/beno/projects/datasets-internal/biqbin/sidon/{filename}', 'w') as f:
    #     json.dump(biqbin_data, f)

In [70]:
with open('/home/beno/projects/datasets-internal/biqbin/k-cluster/1/40/kcluster40_025_10_1.json') as f:
    data = json.load(f)
    
data.keys()

dict_keys(['preprocessing_time', 'qubo', 'offset', 'info'])